In [1]:
import os, shutil, random, math
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision.models import resnet18, ResNet18_Weights

from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [3]:

PATH_140K = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"


PATH_CIFAKE = "/kaggle/input/cifake-real-and-ai-generated-syn"


MERGED_BASE = "/kaggle/working/fft_merged"


In [4]:
for split in ["train", "val", "test"]:
    for cls in ["REAL", "FAKE"]:
        os.makedirs(os.path.join(MERGED_BASE, split, cls), exist_ok=True)

print("Created:", MERGED_BASE)


Created: /kaggle/working/fft_merged


In [5]:
def copy_images(src_dir, dst_dir, limit=None, prefix=""):
    files = [f for f in os.listdir(src_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
    files.sort()
    if limit is not None:
        files = files[:limit]
    for f in tqdm(files, desc=f"Copying {os.path.basename(src_dir)} -> {os.path.basename(dst_dir)}", leave=False):
        src = os.path.join(src_dir, f)
        dst = os.path.join(dst_dir, prefix + f)
        shutil.copy(src, dst)


In [13]:

train_real_140k = os.path.join(PATH_140K, "train", "real")
train_fake_140k = os.path.join(PATH_140K, "train", "fake")

val_real_140k   = os.path.join(PATH_140K, "valid", "real")
val_fake_140k   = os.path.join(PATH_140K, "valid", "fake")

test_real_140k  = os.path.join(PATH_140K, "test", "real")
test_fake_140k  = os.path.join(PATH_140K, "test", "fake")


cifake_train_fake = os.path.join(PATH_CIFAKE, "train", "FAKE")
cifake_test_fake  = os.path.join(PATH_CIFAKE, "test",  "FAKE")


dst_train_real = os.path.join(MERGED_BASE, "train", "REAL")
dst_train_fake = os.path.join(MERGED_BASE, "train", "FAKE")

dst_val_real   = os.path.join(MERGED_BASE, "val", "REAL")
dst_val_fake   = os.path.join(MERGED_BASE, "val", "FAKE")

dst_test_real  = os.path.join(MERGED_BASE, "test", "REAL")
dst_test_fake  = os.path.join(MERGED_BASE, "test", "FAKE")


In [10]:

copy_images(train_real_140k, dst_train_real, prefix="140k_")
copy_images(train_fake_140k, dst_train_fake, prefix="140k_")

copy_images(val_real_140k, dst_val_real, prefix="140k_")
copy_images(val_fake_140k, dst_val_fake, prefix="140k_")

copy_images(test_real_140k, dst_test_real, prefix="140k_")
copy_images(test_fake_140k, dst_test_fake, prefix="140k_")

print("Merged 140k done.")


Merged 140k done.


In [14]:
CIFAKE_TRAIN_ADD = 15000
CIFAKE_VAL_ADD   = 3000
CIFAKE_TEST_ADD  = 3000

copy_images(
    src_dir=cifake_train_fake,
    dst_dir=dst_train_fake,
    limit=CIFAKE_TRAIN_ADD,
    prefix="cifake_"
)

copy_images(
    src_dir=cifake_test_fake,
    dst_dir=dst_val_fake,
    limit=CIFAKE_VAL_ADD,
    prefix="cifake_"
)

copy_images(
    src_dir=cifake_test_fake,
    dst_dir=dst_test_fake,
    limit=CIFAKE_TEST_ADD,
    prefix="cifake_"
)

print("✅ CIFAKE diffusion FAKE images added successfully")


✅ CIFAKE diffusion FAKE images added successfully


In [16]:
def count_dir(d):
    return len([f for f in os.listdir(d) if f.lower().endswith((".jpg",".jpeg",".png"))])

print("TRAIN REAL:", count_dir(dst_train_real), " TRAIN FAKE:", count_dir(dst_train_fake))
print("VAL   REAL:", count_dir(dst_val_real),   " VAL   FAKE:", count_dir(dst_val_fake))
print("TEST  REAL:", count_dir(dst_test_real),  " TEST  FAKE:", count_dir(dst_test_fake))


TRAIN REAL: 50000  TRAIN FAKE: 65000
VAL   REAL: 10000  VAL   FAKE: 13000
TEST  REAL: 10000  TEST  FAKE: 13000


In [17]:
def fft_log_magnitude(gray_img_np):

    f = np.fft.fft2(gray_img_np)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    logmag = np.log1p(mag)
    return logmag

def high_pass_mask(shape, radius_ratio=0.15):

    H, W = shape
    cy, cx = H // 2, W // 2
    r = int(min(H, W) * radius_ratio)
    Y, X = np.ogrid[:H, :W]
    dist = (Y - cy)**2 + (X - cx)**2
    mask = (dist >= r*r).astype(np.float32)
    return mask

def apply_hpf(logmag, radius_ratio=0.15):
    mask = high_pass_mask(logmag.shape, radius_ratio)
    return logmag * mask


In [18]:
def sample_paths(root_dir, n=2000):
    all_files = []
    for cls in ["REAL","FAKE"]:
        cls_dir = os.path.join(root_dir, cls)
        files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
        all_files.extend(files)
    random.shuffle(all_files)
    return all_files[:n]

def compute_fft_mean_std(train_root, img_size=224, radius_ratio=0.15, n=2000):
    paths = sample_paths(train_root, n=n)
    vals = []

    for p in tqdm(paths, desc="Computing FFT mean/std"):
        img = Image.open(p).convert("L").resize((img_size, img_size))
        arr = np.array(img).astype(np.float32)
        logmag = fft_log_magnitude(arr)
        logmag = apply_hpf(logmag, radius_ratio=radius_ratio)
        vals.append(logmag)

    vals = np.stack(vals, axis=0)  
    mean = vals.mean()
    std = vals.std() + 1e-8
    return float(mean), float(std)

TRAIN_ROOT = os.path.join(MERGED_BASE, "train")
fft_mean, fft_std = compute_fft_mean_std(TRAIN_ROOT, n=1500)
print("FFT Mean:", fft_mean, "FFT Std:", fft_std)


Computing FFT mean/std: 100%|██████████| 1500/1500 [00:04<00:00, 345.74it/s]


FFT Mean: 6.154193878173828 FFT Std: 2.088867425918579


In [20]:
class FFTDataset(Dataset):
    def __init__(self, root_dir, img_size=224, radius_ratio=0.15, mean=0.0, std=1.0):
        self.root_dir = root_dir
        self.img_size = img_size
        self.radius_ratio = radius_ratio
        self.mean = mean
        self.std = std

        self.samples = []
        for label, cls in enumerate(["FAKE", "REAL"]):  
            cls_dir = os.path.join(root_dir, cls)
            files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]
            for f in files:
                self.samples.append((os.path.join(cls_dir, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("L").resize((self.img_size, self.img_size))
        arr = np.array(img).astype(np.float32)

        logmag = fft_log_magnitude(arr)
        logmag = apply_hpf(logmag, radius_ratio=self.radius_ratio)


        logmag = (logmag - self.mean) / self.std


        x = torch.tensor(logmag, dtype=torch.float32).unsqueeze(0)
        y = torch.tensor(label, dtype=torch.long)
        return x, y


In [21]:
BATCH_SIZE = 64
NUM_WORKERS = 2  

fft_train_ds = FFTDataset(os.path.join(MERGED_BASE, "train"), mean=fft_mean, std=fft_std)
fft_val_ds   = FFTDataset(os.path.join(MERGED_BASE, "val"),   mean=fft_mean, std=fft_std)
fft_test_ds  = FFTDataset(os.path.join(MERGED_BASE, "test"),  mean=fft_mean, std=fft_std)

fft_train_loader = DataLoader(fft_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
fft_val_loader   = DataLoader(fft_val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
fft_test_loader  = DataLoader(fft_test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("Train size:", len(fft_train_ds), "Val:", len(fft_val_ds), "Test:", len(fft_test_ds))


Train size: 115000 Val: 23000 Test: 23000


In [22]:
fft_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

old_conv = fft_model.conv1
new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
fft_model.conv1 = new_conv

fft_model.fc = nn.Linear(fft_model.fc.in_features, 2)
fft_model = fft_model.to(device)

print("FFT model ready.")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s] 


FFT model ready.


In [23]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, desc="Train", leave=False):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in tqdm(loader, desc="Val/Test", leave=False):
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)

        total_loss += loss.item() * x.size(0)
        pred = out.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += x.size(0)

    return total_loss / total, correct / total


In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(fft_model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 5
best_val_acc = 0.0
SAVE_PATH = "/kaggle/working/best_fft_cifake_hpf.pth"

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(fft_model, fft_train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(fft_model, fft_val_loader, criterion)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"FFT Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"FFT Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(fft_model.state_dict(), SAVE_PATH)
        print("✅ Saved best model:", SAVE_PATH)



Epoch 1/5
FFT Train Loss: 0.4861 | Train Acc: 0.7448
FFT Val   Loss: 0.4318 | Val   Acc: 0.7855
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 2/5
FFT Train Loss: 0.4107 | Train Acc: 0.8024
FFT Val   Loss: 0.4231 | Val   Acc: 0.7931
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 3/5
FFT Train Loss: 0.3613 | Train Acc: 0.8320
FFT Val   Loss: 0.4168 | Val   Acc: 0.8004
✅ Saved best model: /kaggle/working/best_fft_cifake_hpf.pth



Epoch 4/5
FFT Train Loss: 0.2944 | Train Acc: 0.8702
FFT Val   Loss: 0.4595 | Val   Acc: 0.7968



Epoch 5/5
FFT Train Loss: 0.2023 | Train Acc: 0.9157
FFT Val   Loss: 0.5845 | Val   Acc: 0.7843


In [25]:

fft_model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
fft_model.eval()

@torch.no_grad()
def evaluate_metrics(model, loader):
    model.eval()
    all_preds = []
    all_labels = []

    for x, y in tqdm(loader, desc="Metrics", leave=False):
        x = x.to(device)
        out = model(x)
        pred = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(pred.tolist())
        all_labels.extend(y.numpy().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', pos_label=0)  


    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=["Fake","Real"])

    return acc, prec, rec, f1, cm, report

test_acc, test_prec, test_rec, test_f1, cm, report = evaluate_metrics(fft_model, fft_test_loader)

print(f"\nFFT Test Accuracy : {test_acc:.4f}")
print(f"FFT Precision     : {test_prec:.4f}")
print(f"FFT Recall        : {test_rec:.4f}")
print(f"FFT F1-score      : {test_f1:.4f}")
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)



FFT Test Accuracy : 0.8058
FFT Precision     : 0.8650
FFT Recall        : 0.7778
FFT F1-score      : 0.8191

Confusion Matrix:
 [[10111  2889]
 [ 1578  8422]]

Classification Report:
               precision    recall  f1-score   support

        Fake       0.87      0.78      0.82     13000
        Real       0.74      0.84      0.79     10000

    accuracy                           0.81     23000
   macro avg       0.80      0.81      0.80     23000
weighted avg       0.81      0.81      0.81     23000



In [26]:
import torch.nn.functional as F

def predict_fft_image(model, img_path, device, img_size=224, radius_ratio=0.15, mean=0.0, std=1.0):
    img = Image.open(img_path).convert("L").resize((img_size, img_size))
    arr = np.array(img).astype(np.float32)

    logmag = fft_log_magnitude(arr)
    logmag = apply_hpf(logmag, radius_ratio=radius_ratio)
    logmag = (logmag - mean) / std

    x = torch.tensor(logmag, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)  

    model.eval()
    with torch.no_grad():
        out = model(x)
        prob = F.softmax(out, dim=1).cpu().numpy()[0]  
        pred = int(np.argmax(prob))

    label_map = {0: "FAKE", 1: "REAL"}

    return {
        "prediction": label_map[pred],
        "fake_confidence": float(prob[0]),
        "real_confidence": float(prob[1])
    }


In [28]:
img_path = "/kaggle/input/fft-test-images/gemini-image-1.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.5821288228034973
Real confidence: 0.4178711771965027


In [29]:
img_path = "/kaggle/input/fft-test-images/gemini-image-2.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.5597735047340393
Real confidence: 0.4402264952659607


In [30]:
img_path = "/kaggle/input/fft-test-images/gemini-image-3.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.5704056620597839
Real confidence: 0.42959436774253845


In [31]:
img_path = "/kaggle/input/fft-test-images/gemini-image-4.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.8395127654075623
Real confidence: 0.16048727929592133


In [32]:
img_path = "/kaggle/input/fft-test-images/gemini-image-5.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.4601881802082062
Real confidence: 0.5398117899894714


In [33]:
img_path = "/kaggle/input/fft-test-images/gpt-image-1.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.7304527163505554
Real confidence: 0.26954734325408936


In [34]:
img_path = "/kaggle/input/fft-test-images/gpt-image-2.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.06221558526158333
Real confidence: 0.9377843737602234


In [35]:
img_path = "/kaggle/input/fft-test-images/gpt-image-3.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: FAKE
Fake confidence: 0.7382216453552246
Real confidence: 0.2617783844470978


In [36]:
img_path = "/kaggle/input/fft-test-images/gpt-image-4.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.13907578587532043
Real confidence: 0.860924243927002


In [37]:
img_path = "/kaggle/input/fft-test-images/gpt-image-5.png"
result = predict_fft_image(fft_model, img_path, device, mean=fft_mean, std=fft_std)

print("Prediction:", result["prediction"])
print("Fake confidence:", result["fake_confidence"])
print("Real confidence:", result["real_confidence"])


Prediction: REAL
Fake confidence: 0.23271650075912476
Real confidence: 0.7672834992408752


In [1]:
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


fft_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)


old_conv = fft_model.conv1
new_conv = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)
new_conv.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)
fft_model.conv1 = new_conv


fft_model.fc = nn.Linear(fft_model.fc.in_features, 2)

fft_model = fft_model.to(device)


Using device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 163MB/s] 


In [5]:
import os

print("Contents of /kaggle/working:")
for f in os.listdir("/kaggle/working"):
    print(f)


Contents of /kaggle/working:
.virtual_documents


In [44]:
img_path = "/kaggle/input/fft-test-images/gpt-image-2.png"

result = predict_fft_image(fft_model, img_path, device)

print("Prediction:", result["prediction"])
print(f"Fake confidence: {result['fake_confidence']:.4f}")
print(f"Real confidence: {result['real_confidence']:.4f}")


Prediction: REAL
Fake confidence: 0.1556
Real confidence: 0.8444


In [45]:
img_path = "/kaggle/input/fft-test-images/gpt-image-3.png"

result = predict_fft_image(fft_model, img_path, device)

print("Prediction:", result["prediction"])
print(f"Fake confidence: {result['fake_confidence']:.4f}")
print(f"Real confidence: {result['real_confidence']:.4f}")


Prediction: REAL
Fake confidence: 0.0002
Real confidence: 0.9998


In [46]:
img_path = "/kaggle/input/fft-test-images/gpt-image-4.png"

result = predict_fft_image(fft_model, img_path, device)

print("Prediction:", result["prediction"])
print(f"Fake confidence: {result['fake_confidence']:.4f}")
print(f"Real confidence: {result['real_confidence']:.4f}")


Prediction: FAKE
Fake confidence: 0.5192
Real confidence: 0.4808


In [47]:
img_path = "/kaggle/input/fft-test-images/gpt-image-5.png"

result = predict_fft_image(fft_model, img_path, device)

print("Prediction:", result["prediction"])
print(f"Fake confidence: {result['fake_confidence']:.4f}")
print(f"Real confidence: {result['real_confidence']:.4f}")


Prediction: REAL
Fake confidence: 0.0033
Real confidence: 0.9967
